# HAM10000 期末實驗 01：Baseline（SVM、Random Forest）

對應報告參之四、參之五。先執行 `00_資料檢查`，再執行這份。只需要 CPU。

| 實驗 | 內容 | 對應 |
|---|---|---|
| 調參 | 以種子 42 依病灶切分的**驗證集**網格搜尋，選定後固定用於所有種子與切分 | 表 3-2 |
| E1 | 依病灶切分，五個種子 | RQ1 |
| E3 | 依影像切分，五個種子 | RQ3 |
| E3b | 同一份依影像切分的測試集：模型 A（原訓練集）、B（移除與測試集同病灶的影像）、C（移除等量且無關的病灶） | RQ3 |
| E4 | 依病灶與依影像切分的模型預測 ISIC 2018 全部 1,705 張，分析時才套用排除名單 | RQ1、RQ3 |

**可續跑**：每完成一次訓練就存檔。Colab 斷線後重新「全部執行」，已完成的會自動略過。

預估時間：特徵擷取約 10 分鐘，調參約 20–40 分鐘，正式實驗 40 次訓練約 1–2 小時。
> **正式版**：以 Claude 原版與 ChatGPT 審查修正版合併。所有預測都附影像編號，分析時依編號對齊。

## 0. 環境設定

In [ ]:
import sys, subprocess, os
IN_COLAB = 'google.colab' in sys.modules
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'], check=False)

USE_DRIVE = True
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
BASE = os.environ.get('HAM_BASE') or ('/content/drive/MyDrive/ham10000_project' if (IN_COLAB and USE_DRIVE) else './ham10000_project')

In [ ]:
import time, json, random, platform, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from joblib import Parallel, delayed

import skimage
from skimage.feature import hog
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay)

# ===== 實驗設定（報告表 3-2、表 3-3、表 3-5）=====
SEEDS = [42, 43, 44, 45, 46]         # 事先決定，全部回報
TUNE_SEED = 42                       # 調參只用這個種子的驗證集
IMG_SIZE = 64
VAL_RATIO, TEST_RATIO = 0.15, 0.15
RUN_E4 = True                        # 需要 00 筆記本產生的 isic2018_external.csv（訓練時預測全部 1,705 張）

SVM_GRID = {'C': [0.1, 1, 10, 100], 'gamma': ['scale', 0.001, 0.0001]}
RF_GRID = {'max_depth': [None, 10, 20], 'min_samples_leaf': [1, 5, 10],
           'class_weight': ['balanced', 'balanced_subsample']}
RF_TREES = 300

RES_DIR = os.path.join(BASE, 'results_final_baseline')
PRED_DIR = os.path.join(RES_DIR, 'predictions')
CACHE_DIR = os.path.join(BASE, 'cache')
CHECK_DIR = os.path.join(BASE, 'data_check')
for d in [RES_DIR, PRED_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

CLASS_ORDER = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
SPLITS = ['lesion', 'image', 'image_noleak', 'image_ctrl']
SPLIT_LABEL = {'lesion': '依病灶切分', 'image': '依影像切分（模型 A）',
               'image_noleak': '移除與測試集同病灶的影像（模型 B）', 'image_ctrl': '移除等量無關病灶（模型 C）'}
print('結果資料夾：', RES_DIR)

## 1. 讀取 HAM10000

In [ ]:
DATA_DIR = os.environ.get('HAM_DATA_DIR')
if DATA_DIR is None:
    import kagglehub
    DATA_DIR = kagglehub.dataset_download('kmader/skin-cancer-mnist-ham10000')

meta_path, img_paths = None, {}
for root, _, files in os.walk(DATA_DIR):
    for f in files:
        if f == 'HAM10000_metadata.csv' and meta_path is None:
            meta_path = os.path.join(root, f)
        if f.lower().endswith('.jpg'):
            img_paths.setdefault(os.path.splitext(f)[0], os.path.join(root, f))
assert meta_path is not None, '找不到 HAM10000_metadata.csv'
df = pd.read_csv(meta_path)
df['path'] = df['image_id'].map(img_paths)
assert df['path'].notna().all(), '有影像找不到檔案'
assert (df.groupby('lesion_id')['dx'].nunique() == 1).all(), '有病灶對應到多種診斷'
df = df.sort_values('image_id').reset_index(drop=True)
y = df['dx'].values
print(f'影像 {len(df):,} 張，病灶 {df.lesion_id.nunique():,} 個')
SMOKE_MODE = os.environ.get('HAM_SMOKE') == '1'
# 00 以 pHash 候選加 SSIM ≥ 0.90 找到、並經人工確認為同一病灶但病灶編號不同的影像。
# 切分前把它們視為同一個病灶，避免同一病灶被分到訓練集與測試集。原始編號保留在 lesion_id_original
LESION_MERGE = [('HAM_0004919', 'HAM_0000140')]
df['lesion_id_original'] = df['lesion_id']
if not SMOKE_MODE:
    for keep_id, merge_id in LESION_MERGE:
        assert {keep_id, merge_id}.issubset(set(df['lesion_id'])), f'找不到病灶編號 {keep_id} 或 {merge_id}'
        df.loc[df['lesion_id'] == merge_id, 'lesion_id'] = keep_id
    conf = os.path.join(CHECK_DIR, 'ham_confirmed_same_photo.csv')
    assert os.path.exists(conf), '找不到 ham_confirmed_same_photo.csv，請先用正式版 00 執行資料檢查'
    c_ = pd.read_csv(conf)
    found = {frozenset(x) for x in zip(c_['lesion_a'], c_['lesion_b'])}
    assert found == {frozenset(x) for x in LESION_MERGE}, f'00 確認的重複病灶與 LESION_MERGE 不一致：{found}。請人工確認後更新 LESION_MERGE'
print('合併後病灶數：', df['lesion_id'].nunique())

## 2. 特徵擷取（與期中相同：HSV 直方圖 48＋RGB 動差 6＋HOG 1,764＝1,818 維）

In [ ]:
def extract_features(path, size=IMG_SIZE):
    img = Image.open(path).convert('RGB').resize((size, size), Image.BILINEAR)
    rgb = np.asarray(img, dtype=np.float32) / 255.0
    hsv = np.asarray(img.convert('HSV'))
    hist = []
    for c in range(3):
        h, _ = np.histogram(hsv[..., c], bins=16, range=(0, 256))
        hist.append(h / h.sum())
    moments = np.concatenate([rgb.reshape(-1, 3).mean(0), rgb.reshape(-1, 3).std(0)])
    gray = np.asarray(img.convert('L'))
    hog_f = hog(gray, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), feature_vector=True)
    return np.concatenate([*hist, moments, hog_f]).astype(np.float32)

def cached_features(paths, ids, name):
    f, fi = os.path.join(CACHE_DIR, f'{name}.npy'), os.path.join(CACHE_DIR, f'{name}_ids.txt')
    if os.path.exists(f) and os.path.exists(fi) and open(fi).read().split() == list(ids):
        return np.load(f)
    t0 = time.time()
    X = np.vstack(Parallel(n_jobs=-1)(delayed(extract_features)(p) for p in paths))
    np.save(f, X); open(fi, 'w').write('\n'.join(ids))
    print(f'{name}：特徵擷取完成，耗時 {time.time()-t0:.0f} 秒')
    return X

X = cached_features(df['path'], df['image_id'], f'ham_features_{IMG_SIZE}px')
print('HAM10000 特徵：', X.shape)

In [ ]:
# E4 外部資料（ISIC 2018）：訓練時預測全部 1,705 張，排除名單在分析時才套用
# 這樣排除規則之後若有調整，不需要重新訓練
ext = None
if RUN_E4:
    ext_file = os.path.join(CHECK_DIR, 'isic2018_external.csv')
    assert os.path.exists(ext_file), '找不到 isic2018_external.csv，請先執行 00 筆記本，或把 RUN_E4 改成 False'
    ext = pd.read_csv(ext_file)
    assert {'exclude', 'exclude_sensitivity'}.issubset(ext.columns), 'isic2018_external.csv 是舊版，請用正式版 00 重新執行'
    assert ext['image_id'].is_unique and (len(ext) == 1705 or os.environ.get('HAM_SMOKE') == '1'), 'ISIC 2018 影像數不是 1,705 張'
    X_ext = cached_features([os.path.join(BASE, p) for p in ext['path']], ext['image_id'], f'isic2018_all_features_{IMG_SIZE}px')
    y_ext = ext['dx'].values
    print('外部資料（全部）：', X_ext.shape, '正式保留', int((~ext['exclude']).sum()), '張，敏感度分析保留', int((~ext['exclude_sensitivity']).sum()), '張')

## 3. 資料切分（五個種子 × 四種訓練集）

- **lesion**：依病灶切分（正式設定，E1）
- **image**：依影像切分（E3，也是 E3b 的模型 A）
- **image_noleak**：與 image 相同的驗證集與測試集，訓練集移除「病灶也出現在測試集」的影像（E3b 的模型 B）
- **image_ctrl**：與 image 相同的驗證集與測試集，訓練集移除與測試集**無關**的整個病灶，每類移除張數與模型 B 相同（E3b 的模型 C）

模型 B 和 C 的訓練張數與類別組成相同，主要差異在於是否含有與測試影像同病灶的影像，所以 **C − B 是 E3b 的主要比較**。

每個種子同時決定切分與模型隨機性，同一種子下所有模型共用同一份切分。

In [ ]:
holdout = VAL_RATIO + TEST_RATIO
test_share = TEST_RATIO / holdout
assert (df.groupby('lesion_id')['dx'].nunique() == 1).all(), '合併後有病灶對應到多種診斷'
LES_DX = df.groupby('lesion_id')['dx'].first()
LES_SIZE = df.groupby('lesion_id').size()          # 每個病灶在整個資料集的影像數

def split_by_image(seed):
    idx = np.arange(len(df))
    tr, tmp = train_test_split(idx, test_size=holdout, stratify=df['dx'], random_state=seed)
    va, te = train_test_split(tmp, test_size=test_share, stratify=df['dx'].values[tmp], random_state=seed)
    return np.sort(tr), np.sort(va), np.sort(te)

def split_by_lesion(seed):
    les = df.groupby('lesion_id')['dx'].first()
    tr_l, tmp_l = train_test_split(les.index.values, test_size=holdout, stratify=les.values, random_state=seed)
    va_l, te_l = train_test_split(tmp_l, test_size=test_share, stratify=les.loc[tmp_l].values, random_state=seed)
    lid = df['lesion_id'].values
    return (np.where(np.isin(lid, tr_l))[0], np.where(np.isin(lid, va_l))[0], np.where(np.isin(lid, te_l))[0])

def control_removal(seed, tr, va, te, removed_b):
    """模型 C：從訓練集移除與測試集無關的「整個病灶」，每個類別移除的影像數與模型 B 相同。
    優先抽取在整個資料集有多張影像的病灶（模型 B 移除的病灶都屬於這種），不足時再用單張影像的病灶補足。"""
    lid, dx = df['lesion_id'].values, df['dx'].values
    rng = np.random.default_rng(seed)
    test_les, val_les = set(lid[te]), set(lid[va])
    tr_count = pd.Series(lid[tr]).value_counts()              # 每個病灶在訓練集的影像數
    target = pd.Series(dx[removed_b]).value_counts()
    chosen, report = [], []
    for c in CLASS_ORDER:
        need = int(target.get(c, 0))
        if need == 0:
            continue
        cand = sorted(l for l in tr_count.index if l not in test_les and LES_DX[l] == c)
        multi = [l for l in cand if LES_SIZE[l] >= 2]
        single = [l for l in cand if LES_SIZE[l] < 2]
        got, got_multi = 0, 0
        for l in list(rng.permutation(multi)) + list(rng.permutation(single)):
            k = int(tr_count[l])
            if got + k <= need:
                chosen.append(l); got += k
                got_multi += k if LES_SIZE[l] >= 2 else 0
            if got == need:
                break
        b_les = set(lid[removed_b][dx[removed_b] == c]); c_les = set(l for l in chosen if LES_DX[l] == c)
        report.append({'種子': seed, '類別': c, '模型 B 移除影像': need, '模型 C 移除影像': got,
                       '模型 B 移除病灶數': len(b_les), '模型 C 移除病灶數': len(c_les),
                       '模型 C 來自多張影像病灶的影像比例': round(got_multi / got, 3) if got else np.nan,
                       '模型 B 移除病灶也在驗證集的數量': len(b_les & val_les), '模型 C 移除病灶也在驗證集的數量': len(c_les & val_les)})
    removed_c = tr[np.isin(lid[tr], chosen)]
    return np.setdiff1d(tr, removed_c), pd.DataFrame(report)

CONTROL_REPORT = []
def get_splits(seed):
    tr, va, te = split_by_image(seed)
    lid = df['lesion_id'].values
    tr_noleak = tr[~np.isin(lid[tr], lid[te])]
    tr_ctrl, rep = control_removal(seed, tr, va, te, np.setdiff1d(tr, tr_noleak))
    CONTROL_REPORT.append(rep)
    # 模型 B、C 選擇回合用的驗證集：移除與測試集、模型 B 訓練集或模型 C 訓練集同病灶的影像
    # 這樣驗證集的病灶不會出現在任一方的訓練集，兩個模型的回合選擇不受訓練與驗證重疊程度不同的影響
    va_clean = va[~(np.isin(lid[va], lid[te]) | np.isin(lid[va], lid[tr_noleak]) | np.isin(lid[va], lid[tr_ctrl]))]
    return {'lesion': split_by_lesion(seed), 'image': (tr, va, te),
            'image_noleak': (tr_noleak, va_clean, te), 'image_ctrl': (tr_ctrl, va_clean, te)}

def seen_mask(seed):
    # 依影像切分的測試集中，哪些影像的病灶也出現在（模型 A 的）訓練集。模型 C 保留了這些影像
    tr, _, te = SPLITS_BY_SEED[seed]['image']
    lid = df['lesion_id'].values
    return np.isin(lid[te], lid[tr])

SPLITS_BY_SEED = {s: get_splits(s) for s in SEEDS}

# 切分指紋：結果資料夾若存有不同切分（例如合併病灶前）產生的結果，直接停止，避免新舊結果混用
import hashlib
_h = hashlib.sha256(repr(LESION_MERGE).encode())
for s in SEEDS:
    for name in ['lesion', 'image', 'image_noleak', 'image_ctrl']:
        for part in SPLITS_BY_SEED[s][name]:
            _h.update(np.asarray(part, dtype=np.int64).tobytes())
SPLIT_FINGERPRINT = _h.hexdigest()[:16]
_fp = os.path.join(RES_DIR, 'split_fingerprint.txt')
if os.path.exists(_fp):
    assert open(_fp).read().strip() == SPLIT_FINGERPRINT, f'{RES_DIR} 內的結果來自不同的資料切分。請把舊資料夾改名備份後重新執行'
else:
    assert not any(f.endswith(('.csv', '.json')) for f in os.listdir(RES_DIR)), f'{RES_DIR} 已有結果但沒有切分指紋，可能是舊版產出。請把舊資料夾改名備份後重新執行'
    open(_fp, 'w').write(SPLIT_FINGERPRINT)
print('切分指紋：', SPLIT_FINGERPRINT)
control_report = pd.concat(CONTROL_REPORT, ignore_index=True)

# 檢查：模型 B 訓練集與測試集沒有同病灶，模型 C 與 B 的類別張數相同
for s in SEEDS:
    lid = df['lesion_id'].values
    trb, _, te = SPLITS_BY_SEED[s]['image_noleak']
    trc = SPLITS_BY_SEED[s]['image_ctrl'][0]
    assert not np.isin(lid[trb], lid[te]).any(), f'種子 {s}：模型 B 仍有測試病灶'
    assert set(lid[np.setdiff1d(SPLITS_BY_SEED[s]['image'][0], trc)]).isdisjoint(set(lid[te])), f'種子 {s}：模型 C 移除到測試病灶'
    vab = SPLITS_BY_SEED[s]['image_noleak'][1]
    assert not np.isin(lid[vab], lid[te]).any(), f'種子 {s}：模型 B、C 的驗證集仍有測試病灶'
    assert not np.isin(lid[vab], lid[trb]).any() and not np.isin(lid[vab], lid[trc]).any(), f'種子 {s}：模型 B、C 的驗證集仍有訓練病灶'
    vc = pd.Series(df['dx'].values[vab]).value_counts().reindex(CLASS_ORDER).fillna(0).astype(int)
    print(f'種子 {s} 模型 B、C 驗證集各類影像數：', vc.to_dict())
    assert SMOKE_MODE or (vc >= 3).all(), f'種子 {s}：模型 B、C 的驗證集有類別少於 3 張，無法可靠選擇回合'
r = control_report
bad = r[r['模型 B 移除影像'] != r['模型 C 移除影像']]
# 模型 B 與 C 每個類別的移除張數必須完全相同，否則 C − B 的比較前提不成立，停止執行
assert bad.empty, '模型 C 無法完全配對，請勿開始 E3b 訓練：\n' + bad.to_string(index=False)

rows, assign = [], df[['image_id', 'lesion_id', 'dx']].copy()
for s in SEEDS:
    for name, (tr, va, te) in SPLITS_BY_SEED[s].items():
        lid = df['lesion_id'].values
        rows.append({'種子': s, '切分': name, '訓練影像': len(tr), '驗證影像': len(va), '測試影像': len(te),
                     '測試集病灶出現在訓練集的比例': round(np.isin(lid[te], lid[tr]).mean(), 4)})
        if name in ('lesion', 'image'):
            col = np.full(len(df), '', dtype=object); col[tr], col[va], col[te] = 'train', 'val', 'test'
            assign[f'seed{s}_{name}'] = col
        else:
            assign[f'seed{s}_{name}_removed'] = np.isin(np.arange(len(df)), np.setdiff1d(SPLITS_BY_SEED[s]['image'][0], tr))
split_summary = pd.DataFrame(rows)
split_summary.to_csv(os.path.join(RES_DIR, 'split_summary.csv'), index=False, encoding='utf-8-sig')
assign.to_csv(os.path.join(RES_DIR, 'split_assignment.csv'), index=False)
control_report.to_csv(os.path.join(RES_DIR, 'E3b_control_matching.csv'), index=False, encoding='utf-8-sig')
print(split_summary.to_string(index=False))

## 4. 評估函式

In [ ]:
MEL = 'mel'
def evaluate(y_true, y_pred, labels=None):
    labels = CLASS_ORDER if labels is None else labels    # 正式七類評估固定七類，E3b 子群另傳當次出現的類別
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average='macro', zero_division=0)
    mp, mr, _, _ = precision_recall_fscore_support(y_true, y_pred, labels=[MEL], zero_division=0)
    return {'Accuracy': accuracy_score(y_true, y_pred), 'Precision (macro)': p, 'Recall (macro)': r,
            'F1-score (macro)': f, 'Balanced Accuracy': r,
            'Melanoma Recall': mr[0] if (y_true == MEL).any() else np.nan,          # 沒有黑色素瘤時不計算
            'Melanoma Precision': mp[0],                                          # 沒有判為黑色素瘤時記為 0
            'Melanoma FN': int(((y_true == MEL) & (y_pred != MEL)).sum()),
            'Melanoma FP': int(((y_true != MEL) & (y_pred == MEL)).sum())}

def make_model(name, params, seed):
    if name == 'SVM':
        return make_pipeline(StandardScaler(), SVC(kernel='rbf', class_weight='balanced', **params))
    return RandomForestClassifier(n_estimators=RF_TREES, n_jobs=-1, random_state=seed, **params)

## 5. 調參（只用種子 42 依病灶切分的驗證集）

選出驗證集 macro-F1 最高的組合，存成 `best_params.json`，之後所有種子與切分都用這組，不再更動。測試集完全不參與。

In [ ]:
PARAM_FILE = os.path.join(RES_DIR, 'best_params.json')
TUNE_FILE = os.path.join(RES_DIR, 'tuning_validation.csv')
if os.path.exists(PARAM_FILE):
    BEST = json.load(open(PARAM_FILE))
    print('讀取已選定的參數：', BEST)
else:
    tr, va, _ = SPLITS_BY_SEED[TUNE_SEED]['lesion']
    done = pd.read_csv(TUNE_FILE) if os.path.exists(TUNE_FILE) else pd.DataFrame(columns=['模型', '參數'])
    for name, grid in [('SVM', SVM_GRID), ('Random Forest', RF_GRID)]:
        for combo in itertools.product(*grid.values()):
            params = dict(zip(grid.keys(), combo))
            key = json.dumps(params)
            if ((done['模型'] == name) & (done['參數'] == key)).any():
                continue
            t0 = time.time()
            m = make_model('SVM' if name == 'SVM' else 'RF', params, TUNE_SEED).fit(X[tr], y[tr])
            pred = m.predict(X[va])
            row = {'模型': name, '參數': key, '驗證集 macro-F1': f1_score(y[va], pred, average='macro'),
                   '驗證集黑色素瘤召回率': evaluate(y[va], pred)['Melanoma Recall'], '訓練秒數': round(time.time() - t0, 1)}
            done = pd.concat([done, pd.DataFrame([row])], ignore_index=True)
            done.to_csv(TUNE_FILE, index=False, encoding='utf-8-sig')
            print(f"{name} {params} → val macro-F1 {row['驗證集 macro-F1']:.3f}（{row['訓練秒數']} 秒）")
    BEST = {}
    for name in ['SVM', 'Random Forest']:
        d = done[done['模型'] == name]
        BEST[name] = json.loads(d.loc[d['驗證集 macro-F1'].astype(float).idxmax(), '參數'])
    json.dump(BEST, open(PARAM_FILE, 'w'), ensure_ascii=False, indent=1)
    print('選定參數：', BEST)

tune = pd.read_csv(TUNE_FILE)
print(tune.sort_values(['模型', '驗證集 macro-F1'], ascending=[True, False]).groupby('模型').head(3).round(3).to_string(index=False))

## 6. 正式實驗：5 個種子 × 4 種訓練集 × 2 個模型（共 40 次訓練）

每次訓練保存驗證集、測試集與外部資料的預測結果，之後的分析都從這些檔案計算。

In [ ]:
RUNS_CSV = os.path.join(RES_DIR, 'all_runs.csv')
MODELS = ['SVM', 'Random Forest']

def run_if_needed(model_name, split, seed):
    done = pd.read_csv(RUNS_CSV) if os.path.exists(RUNS_CSV) else pd.DataFrame(columns=['模型', '切分', '種子'])
    if ((done['模型'] == model_name) & (done['切分'] == split) & (done['種子'] == seed)).any():
        pred_path = os.path.join(PRED_DIR, f'pred__{model_name.replace(" ", "")}__{split}__seed{seed}.npz')
        if not os.path.exists(pred_path):
            raise RuntimeError(f'CSV 宣稱已完成，但預測檔缺失：{pred_path}。請核對後才續跑。')
        with np.load(pred_path) as old:
            if not {'val_image_id','test_image_id'}.issubset(old.files):
                raise RuntimeError(f'舊版預測檔缺少影像 ID：{pred_path}。不可直接沿用，請重新產生。')
            if ext is not None and ('ext_image_id' not in old.files or set(old['ext_image_id']) != set(ext['image_id'])):
                raise RuntimeError(f'預測檔沒有涵蓋全部 1,705 張外部影像：{pred_path}。請重新產生。')
        return
    tr, va, te = SPLITS_BY_SEED[seed][split]
    t0 = time.time()
    m = make_model('SVM' if model_name == 'SVM' else 'RF', BEST[model_name], seed).fit(X[tr], y[tr])
    secs = time.time() - t0
    preds = {'val': m.predict(X[va]), 'test': m.predict(X[te]),
             'val_image_id': df.image_id.iloc[va].to_numpy(dtype=str),
             'test_image_id': df.image_id.iloc[te].to_numpy(dtype=str)}
    if ext is not None:
        preds['ext'] = m.predict(X_ext)
        preds['ext_image_id'] = ext.image_id.to_numpy(dtype=str)
    np.savez(os.path.join(PRED_DIR, f'pred__{model_name.replace(" ", "")}__{split}__seed{seed}.npz'), **preds)
    row = {'模型': model_name, '切分': split, '種子': seed, '訓練影像數': len(tr),
           '驗證集 macro-F1': f1_score(y[va], preds['val'], average='macro')}
    row.update(evaluate(y[te], preds['test']))
    row['訓練秒數'] = round(secs, 1)
    pd.concat([done, pd.DataFrame([row])], ignore_index=True).to_csv(RUNS_CSV, index=False, encoding='utf-8-sig')
    print(f"種子 {seed} | {SPLIT_LABEL[split]} | {model_name}：test macro-F1 {row['F1-score (macro)']:.3f}（{secs:.0f} 秒）")

for seed in SEEDS:                       # 一個種子完整跑完再跑下一個
    for split in SPLITS:
        for model_name in MODELS:
            run_if_needed(model_name, split, seed)
runs = pd.read_csv(RUNS_CSV)
print('完成訓練次數：', len(runs), '／', len(SEEDS) * len(SPLITS) * len(MODELS))

## 7. 彙總工具

- 一般結果：五個種子的「平均值 ± 標準差」
- 比較兩種設定：先算同一種子下的差值，回報各種子差值、差值的平均值 ± 標準差，以及差值為正的種子數。只作描述，不據此宣稱「相同」或「顯著」。
- 所有預測都依影像編號對齊，編號不符會直接報錯。

In [ ]:
def fmt(s):
    s = pd.Series(s, dtype=float).dropna()
    return f'{s.mean():.3f} ± {s.std(ddof=1):.3f}' if len(s) > 1 else (f'{s.mean():.3f}' if len(s) else '')

def paired(a, b, prefix=''):
    d = (pd.Series(a, dtype=float) - pd.Series(b, dtype=float)).dropna()
    return {f'{prefix}差值（平均 ± 標準差）': fmt(d), f'{prefix}差值為正的種子數': f'{int((d > 0).sum())}/{len(d)}',
            f'{prefix}各種子差值': '、'.join(f'{v:+.3f}' for v in d)}

METRICS = ['Accuracy', 'F1-score (macro)', 'Balanced Accuracy', 'Melanoma Recall', 'Melanoma Precision']
def per_seed(model, split, metric):
    r = runs[(runs['模型'] == model) & (runs['切分'] == split)].set_index('種子')[metric]
    return r.reindex(SEEDS)

def load_pred(model, split, seed):
    return np.load(os.path.join(PRED_DIR, f'pred__{model.replace(" ", "")}__{split}__seed{seed}.npz'), allow_pickle=True)

def pred_for(model, split, seed, part, ids):
    # 依影像編號取出預測，part 為 'val'、'test' 或 'ext'
    p = load_pred(model, split, seed)
    pos = pd.Series(np.arange(len(p[f'{part}_image_id'])), index=p[f'{part}_image_id'])
    missing = set(ids) - set(pos.index)
    assert not missing, f'預測檔缺少 {len(missing)} 張影像：{model} {split} {seed} {part}'
    return p[part][pos.loc[list(ids)].values]

def test_ids(split, seed):
    return df['image_id'].values[SPLITS_BY_SEED[seed][split][2]]

## 8. E1：依病灶切分的 Baseline 效能（表 4-1 的 Baseline 部分）

In [ ]:
e1 = pd.DataFrame({m: {k: fmt(per_seed(m, 'lesion', k)) for k in METRICS} for m in MODELS}).T
e1.index.name = '模型'
e1.to_csv(os.path.join(RES_DIR, 'E1_baseline_mean_sd.csv'), encoding='utf-8-sig')
print(e1.to_string())

## 9. E3：依影像切分 − 依病灶切分（同種子配對）

In [ ]:
e3 = []
for m in MODELS:
    for k in ['F1-score (macro)', 'Melanoma Recall', 'Balanced Accuracy']:
        e3.append({'模型': m, '指標': k, '依病灶切分': fmt(per_seed(m, 'lesion', k)),
                   '依影像切分': fmt(per_seed(m, 'image', k)), **paired(per_seed(m, 'image', k), per_seed(m, 'lesion', k))})
e3 = pd.DataFrame(e3)
e3.to_csv(os.path.join(RES_DIR, 'E3_baseline_paired.csv'), index=False, encoding='utf-8-sig')
print(e3.to_string(index=False))

## 10. E3b：同一份測試集比較模型 A、B、C

- **主要比較 C − B**：兩者訓練張數與類別組成相同，主要差異在是否含有與測試影像同病灶的影像，移除的病灶數與影像難度仍可能不同。
- **輔助比較 A − B**：原始依影像切分與移除重疊後的差距，含有訓練量減少的影響。
- **分群**：依測試影像的病灶是否出現在模型 A 的訓練集分成兩群。若差距來自資料洩漏，C − B 應主要出現在「病灶出現過」的一群。
- 兩群的 macro-F1 只用「兩群都有的類別」計算，並列出類別數。不同群的分數絕對值不直接比較。
- 模型 B、C 的驗證集已移除與測試集或兩者訓練集同病灶的影像（Baseline 不用驗證集選模型，這只影響 CNN）。
- C − B 是控制訓練影像數與類別組成後的比較，B、C 移除的病灶組成仍可能不同（見 `E3b_control_matching.csv`），不視為純粹的因果效果。

In [ ]:
E3B_MODELS = {'A': 'image', 'B': 'image_noleak', 'C': 'image_ctrl'}
E3B_METRICS = ['F1-score (macro)', 'Melanoma Recall', 'Accuracy']

def e3b_long_table(models, getter, y_all, evaluate_fn, class_list):
    out = []
    for m in models:
        for seed in SEEDS:
            te = SPLITS_BY_SEED[seed]['image'][2]
            ids = df['image_id'].values[te]
            yt_all = y_all[te]
            seen = seen_mask(seed)
            common = sorted(set(yt_all[seen]) & set(yt_all[~seen]))
            preds = {k: getter(m, sp, seed, ids) for k, sp in E3B_MODELS.items()}
            for group, mask, labels in [('全部', np.ones(len(te), bool), class_list),
                                        ('病灶出現過', seen, common), ('病灶未出現過', ~seen, common)]:
                yt = yt_all[mask]
                res = {k: evaluate_fn(yt, p[mask], labels=labels) for k, p in preds.items()}
                for metric in E3B_METRICS:
                    out.append({'模型': m, '群組': group, '種子': seed, '指標': metric, '影像數': int(mask.sum()),
                                '計算類別數': len(labels), '計算類別': '、'.join(map(str, labels)), **{k: res[k][metric] for k in E3B_MODELS}})
    return pd.DataFrame(out)

def e3b_summary(long):
    rows = []
    for (m, g, metric), d in long.groupby(['模型', '群組', '指標'], sort=False):
        d = d.set_index('種子')
        rows.append({'模型': m, '群組': g, '指標': metric, '影像數（平均）': round(d['影像數'].mean()),
                     '計算類別數': int(d['計算類別數'].min()),
                     '模型 A': fmt(d['A']), '模型 B': fmt(d['B']), '模型 C': fmt(d['C']),
                     **paired(d['C'], d['B'], 'C−B '), **paired(d['A'], d['B'], 'A−B ')})
    return pd.DataFrame(rows)

e3b_long = e3b_long_table(MODELS, lambda m, sp, seed, ids: pred_for(m, sp, seed, 'test', ids), y, evaluate, CLASS_ORDER)
e3b_long.to_csv(os.path.join(RES_DIR, 'E3b_baseline_per_seed.csv'), index=False, encoding='utf-8-sig')
e3b = e3b_summary(e3b_long)
e3b.to_csv(os.path.join(RES_DIR, 'E3b_baseline_summary.csv'), index=False, encoding='utf-8-sig')
removed = split_summary.pivot(index='種子', columns='切分', values='訓練影像')
print('模型 B、C 移除的訓練影像數：', (removed['image'] - removed['image_noleak']).to_dict(), (removed['image'] - removed['image_ctrl']).to_dict())
print(e3b.to_string(index=False))

## 11. E4：外部資料（ISIC 2018）

- **正式結果**：只排除影像編號相同、IL 病灶編號相同，或 SSIM 確認為同一張照片的影像。
- **敏感度分析**：另外排除所有 pHash 候選，確認這個決定是否影響結論。
- 若 00 的稽核未通過（metadata 不完整或 IL 編號無法一對一），這一節會跳過。

In [ ]:
E4_SETS = {'正式': 'exclude', '敏感度分析': 'exclude_sensitivity'}

def e4_rows(models, getter, y_ext_all, evaluate_fn):
    rows = []
    for set_name, col in E4_SETS.items():
        keep = ~ext[col].values
        ids = ext['image_id'].values[keep]
        groups = [('全部', np.ones(keep.sum(), bool))]
        srcs = ext['source'].values[keep]
        if len(set(srcs)) > 1:
            groups += [(g, srcs == g) for g in ['same', 'different'] if (srcs == g).any()]
        for m in models:
            for sp in ['lesion', 'image']:
                for g, mask in groups:
                    yt = y_ext_all[keep][mask]
                    labels = CLASS_ORDER                     # 固定七類，兩種排除方案才能直接比較
                    vals = [evaluate_fn(yt, getter(m, sp, seed, ids)[mask], labels=labels) for seed in SEEDS]
                    rows.append({'資料集': set_name, '模型': m, '訓練切分': sp, '外部資料群組': g, '影像數': int(mask.sum()),
                                 '計算類別數': len(labels), '各類影像數': '、'.join(f'{c}:{int((yt == c).sum())}' for c in CLASS_ORDER),
                                 **{k: fmt([v[k] for v in vals]) for k in ['F1-score (macro)', 'Balanced Accuracy', 'Melanoma Recall']}})
    return pd.DataFrame(rows)

audit = json.load(open(os.path.join(CHECK_DIR, 'e4_audit_status.json'), encoding='utf-8')) if ext is not None else {}
if ext is not None and audit.get('e4_ready_for_provisional_evaluation') is True:
    e4 = e4_rows(MODELS, lambda m, sp, seed, ids: pred_for(m, sp, seed, 'ext', ids), y_ext, evaluate)
    e4.to_csv(os.path.join(RES_DIR, 'E4_baseline_external.csv'), index=False, encoding='utf-8-sig')
    print(e4.to_string(index=False))
elif ext is not None:
    print('⚠ E4 稽核未通過，略過 E4 分析。請先檢查 00 的 e4_audit_status.json')

## 12. 混淆矩陣（依病灶切分，五個種子的測試集合併）

In [ ]:
best_model = max(MODELS, key=lambda m: per_seed(m, 'lesion', 'F1-score (macro)').mean())
cm = sum(confusion_matrix(y[SPLITS_BY_SEED[s]['lesion'][2]], pred_for(best_model, 'lesion', s, 'test', test_ids('lesion', s)), labels=CLASS_ORDER) for s in SEEDS)
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
ConfusionMatrixDisplay(cm, display_labels=CLASS_ORDER).plot(ax=axes[0], cmap='Blues', colorbar=False, values_format='d')
ConfusionMatrixDisplay(cm / cm.sum(1, keepdims=True), display_labels=CLASS_ORDER).plot(ax=axes[1], cmap='Blues', colorbar=False, values_format='.2f')
for ax, t in zip(axes, ['Counts (5 seeds pooled)', 'Normalized by true class']):
    ax.set_title(f'{best_model} — {t}'); ax.set_xlabel('Predicted class'); ax.set_ylabel('True class')
plt.tight_layout(); plt.savefig(os.path.join(RES_DIR, 'fig_baseline_confusion_matrix.png'), dpi=300); plt.show()

## 13. 實驗環境（表 3-5）

In [ ]:
env = {'執行平台': 'Google Colab' if IN_COLAB else platform.platform(),
       '硬體': f'CPU（{os.cpu_count()} 核心）',
       '程式語言': 'Python ' + platform.python_version(),
       '主要套件與版本': f'scikit-learn {sklearn.__version__}、scikit-image {skimage.__version__}、NumPy {np.__version__}、pandas {pd.__version__}',
       '隨機種子': '、'.join(map(str, SEEDS)),
       '選定參數': json.dumps(BEST, ensure_ascii=False)}
pd.Series(env).to_csv(os.path.join(RES_DIR, 'environment_baseline.csv'), header=['規格'], encoding='utf-8-sig')
print(pd.Series(env).to_string())

## 14. 輸出檔案對照報告

| 檔案 | 用在報告哪裡 |
|---|---|
| `tuning_validation.csv`、`best_params.json` | 表 3-2 Baseline 選定參數 |
| `split_summary.csv`、`split_assignment.csv` | 表 3-1、附錄 C（每個種子的切分名單） |
| `E1_baseline_mean_sd.csv` | 表 4-1 Baseline 部分（CNN 筆記本會自動合併） |
| `E3_baseline_paired.csv` | E3 結果 |
| `E3b_baseline_summary.csv`、`E3b_baseline_per_seed.csv`、`E3b_control_matching.csv` | E3b 結果與模型 C 的配對紀錄 |
| `E4_baseline_external.csv` | E4 結果（正式與敏感度分析） |
| `all_runs.csv`、`predictions/` | 完整紀錄，上傳 GitHub |